# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kishan992/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*



**Task Type:** Classification with Probability Scoring

**Which one and why:**
We are framing this project as a **Binary Classification** task paired with **Probability Scoring**.

* **Supervised Learning:** Because we have historical performance data (clicks, impressions, position), we can teach the model to recognize pages that are dropping in traffic.
* **Classification (1 vs 0):** The model assigns a clear binary label to each web page:
  * **1 = Decaying** (Traffic is dropping; page needs a content refresh)
  * **0 = Stable / Growing** (Traffic is healthy; no immediate action needed)
* **Probability Scoring:** A simple yes/no isn't enough because an editorial team cannot update hundreds of pages at once. The model generates a risk score between **0.0 and 1.0**. This allows us to rank pages by urgency so the team knows which high-value pages to fix first.

In [13]:
# Section 1 Code: Define ML Task Parameters
task_type = "Binary Classification with Probability Scoring"
model_goal = "Predict web page traffic decay risk (0.0 to 1.0 probability)"

print(f"Task Type: {task_type}")
print(f"Model Goal: {model_goal}")

Task Type: Binary Classification with Probability Scoring
Model Goal: Predict web page traffic decay risk (0.0 to 1.0 probability)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


**What we are predicting:**
We are predicting a binary target column called `is_decaying` (1 or 0), which flags whether a page is about to lose traffic and requires a content refresh.

**Where the label comes from:**
This label comes from a **defined rule based on observed outcomes** (a quantitative proxy), using historical search performance data.

**The Labeling Rule (aligned with our 90-day window):**
* We use **90 days of historical search data** (impressions, average rank position, clicks) as our features to train the model.
* We then measure the traffic movement entering the next evaluation window:
  * **Label = 1 (Decaying):** If a page experiences a **≥ 20% drop in clicks** (or sustained position drop) entering the future test window compared to its prior 90-day baseline.
  * **Label = 0 (Stable/Growing):** If the page maintains or increases its traffic performance.

In [14]:
import pandas as pd

# Section 2 Code: Demonstrating the Target Labeling Proxy Rule
demo_data = pd.DataFrame({
    'url_path': ['/page-a', '/page-b', '/page-c'],
    'baseline_clicks_30d': [400, 283, 133],  # Monthly baseline (90d / 3)
    'eval_clicks_30d': [250, 310, 90]        # Evaluation window clicks
})

# Calculate percentage drop cleanly: (New - Baseline) / Baseline
demo_data['pct_change'] = (demo_data['eval_clicks_30d'] - demo_data['baseline_clicks_30d']) / demo_data['baseline_clicks_30d']

# Apply proxy rule: 1 if traffic dropped by 20% or more (-0.20), else 0
demo_data['is_decaying'] = (demo_data['pct_change'] <= -0.20).astype(int)

demo_data[['url_path', 'baseline_clicks_30d', 'eval_clicks_30d', 'pct_change', 'is_decaying']]

,url_path,baseline_clicks_30d,eval_clicks_30d,pct_change,is_decaying
0,/page-a,400,250,-0.375000,1
1,/page-b,283,310,0.095406,0
2,/page-c,133,90,-0.323308,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*


**Primary Metric:** Precision@Top 100

**What number means 'good'?**
* **Target: 85%+ Precision@Top 100** (If the model flags 100 pages for refresh, at least 85 must actually be decaying).
* **Baseline to beat:** 50%.

**Why this metric:**
The editorial team has limited time and can only update ~100 articles per month. High precision ensures we don't waste their time sending healthy pages to be rewritten.

In [15]:
# Section 3 Code: Demonstrating Precision@K Evaluation & Baseline Comparison
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score

# Simulating 100 web pages in our dataset
np.random.seed(42)
y_true = np.random.choice([0, 1], size=100, p=[0.8, 0.2])  # 20% actual decay rate
random_predicted_scores = np.random.uniform(0, 1, size=100) # Random guessing scores

# Build evaluation dataframe
eval_df = pd.DataFrame({'true_decay': y_true, 'predicted_score': random_predicted_scores})

# Select Top 20 highest priority recommendations from random scores
top_20 = eval_df.sort_values(by='predicted_score', ascending=False).head(20)

# Calculate precision on top 20 recommendations
precision_at_20 = precision_score(top_20['true_decay'], (top_20['predicted_score'] >= 0.5).astype(int), zero_division=0)

# Print explicit breakdown and explanation in the output
print("=" * 70)
print("SECTION 3 EVALUATION: RANDOM BASELINE vs. ML TARGET")
print("=" * 70)
print(f"• Total Pages Evaluated           : {len(eval_df)}")
print(f"• Top Priority List Capacity (K)   : 20 pages")
print(f"• True Decaying Pages in Top 20   : {top_20['true_decay'].sum()} out of 20")
print(f"• Random Baseline Precision@Top20 : {precision_at_20:.1%}")
print("-" * 70)
print("EXPLANATION OF OUTPUT:")
print("1. Why is Precision so low (15.0%)?")
print("   Because this baseline uses random guessing. Without ML features,")
print("   most pages flagged for refresh are healthy (false positives).")
print("\n2. What is our goal for the real ML Model?")
print("   Our trained model will use search trends (clicks, ranks, impressions)")
print("   to achieve 85%+ Precision, ensuring editor time isn't wasted.")
print("=" * 70)

SECTION 3 EVALUATION: RANDOM BASELINE vs. ML TARGET
• Total Pages Evaluated           : 100
• Top Priority List Capacity (K)   : 20 pages
• True Decaying Pages in Top 20   : 3 out of 20
• Random Baseline Precision@Top20 : 15.0%
----------------------------------------------------------------------
EXPLANATION OF OUTPUT:
1. Why is Precision so low (15.0%)?
   Because this baseline uses random guessing. Without ML features,
   most pages flagged for refresh are healthy (false positives).

2. What is our goal for the real ML Model?
   Our trained model will use search trends (clicks, ranks, impressions)
   to achieve 85%+ Precision, ensuring editor time isn't wasted.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*


**What is one row in this DataFrame?**
One row represents **one unique web page URL** aggregated over our baseline analysis window.

**Why this unit?**
Content refresh decisions (updating text, fixing SEO titles, re-optimizing keywords) are made at the individual page level. Therefore, both our input features (clicks, impressions, position trends) and our target output (`is_decaying`) must be measured per URL.

In [16]:
import duckdb
import pandas as pd
from datasets import load_dataset

# 1. Load the starter dataset directly from FlyRank's Hugging Face repository
ds = load_dataset("FlyRank/internship-starter", split="train[:5000]")
df_raw = ds.to_pandas()

# 2. Use DuckDB to aggregate search metrics to our unit of analysis (1 row = 1 Content ID)
con = duckdb.connect()
df_real = con.execute("""
    SELECT
        content_id,
        CAST(AVG(impressions_90d) AS INT) AS total_impressions_90d,
        CAST(AVG(impressions_last_30d) AS INT) AS impressions_last_30d,
        CAST(AVG(impressions_prev_30d) AS INT) AS impressions_prev_30d,
        MAX(position_tier) AS position_tier
    FROM df_raw
    GROUP BY content_id
    LIMIT 10
""").df()

# 3. Print explanatory summary output
print("=" * 65)
print("UNIT OF ANALYSIS SUMMARY (REAL DATA FROM FLYRANK HUGGING FACE)")
print("=" * 65)
print(f"• Unit of Analysis   : 1 Row = 1 Unique Content Item (content_id)")
print(f"• Dataset Shape      : {df_real.shape[0]} rows (content items), {df_real.shape[1]} columns")
print("=" * 65)
print("\nFirst 5 Rows from FlyRank Dataset:")

df_real.head()

UNIT OF ANALYSIS SUMMARY (REAL DATA FROM FLYRANK HUGGING FACE)
• Unit of Analysis   : 1 Row = 1 Unique Content Item (content_id)
• Dataset Shape      : 10 rows (content items), 5 columns

First 5 Rows from FlyRank Dataset:


,content_id,total_impressions_90d,impressions_last_30d,impressions_prev_30d,position_tier
0,content_304f48230142,3803,578,987,striking
1,content_331d6c4de07b,11751,3626,4206,page_1
2,content_5e6c160719bc,32574,5696,13828,page_3_5
3,content_d8ee6cc6d642,20919,7665,6441,top_3
4,content_42fb2cad9ecf,7228,5127,1357,page_1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


**Why an `if/else` rule fails:**
* **Confounding Variables & Seasonality:** A simple rule like `if clicks drop > 20% then decaying` fails because traffic drops often happen due to holidays, seasonality, or site-wide technical bugs—not content decay.
* **Non-Linear Interactions:** Traffic decay depends on a complex combination of features: SERP position shifts, CTR trends, impression volume, and competition. An `if` statement cannot dynamically weigh these overlapping signals.
* **Granular Priority Scoring:** A rule-based system only gives binary flags (Yes/No), while Machine Learning provides continuous probability scores (0.0 to 1.0). This allows the content team to rank and address pages by exact risk severity.

In [17]:
# Section 5 Code: Demonstrating where Heuristic Rules fail compared to ML
import pandas as pd

# Creating sample edge cases where simple IF-rules fail
df_edge_cases = pd.DataFrame({
    'content_id': ['page_1_holiday', 'page_2_real_decay', 'page_3_position_drop'],
    'clicks_drop_pct': [-0.35, -0.25, -0.10],      # Drops in traffic
    'impressions_drop_pct': [-0.30, +0.05, -0.05], # Impression trends
    'avg_position_change': [0.0, +4.2, +3.5],      # Position drops (higher rank = worse)
    'actual_decay_reason': ['Site Seasonality', 'Content Decay / Outdated', 'Algorithm / Ranking Shift']
})

# Apply simple Rule-Based logic: "If clicks dropped > 20%, flag as decaying"
df_edge_cases['rule_prediction'] = (df_edge_cases['clicks_drop_pct'] <= -0.20).astype(int)

# Print comparison
print("=" * 70)
print("SECTION 5: HEURISTIC RULE vs. COMPLEXITY BREAKDOWN")
print("=" * 70)
print("• Simple Rule Logic : Flag if clicks drop <= -20%")
print("=" * 70)
print("\nEdge Case Analysis:")
for idx, row in df_edge_cases.iterrows():
    print(f"\n[Content ID]: {row['content_id']}")
    print(f"  - Metrics     : Click Change: {row['clicks_drop_pct']:.0%}, Position Change: +{row['avg_position_change']}")
    print(f"  - Simple Rule : Flagged as Decaying = {bool(row['rule_prediction'])}")
    print(f"  - Ground Truth: {row['actual_decay_reason']}")
    print(f"  - ML Advantage: {'FALSE POSITIVE (Wasted refresh effort)' if row['rule_prediction'] == 1 and row['actual_decay_reason'] != 'Content Decay / Outdated' else 'Correctly identified multi-feature signal'}")

print("=" * 70)

SECTION 5: HEURISTIC RULE vs. COMPLEXITY BREAKDOWN
• Simple Rule Logic : Flag if clicks drop <= -20%

Edge Case Analysis:

[Content ID]: page_1_holiday
  - Metrics     : Click Change: -35%, Position Change: +0.0
  - Simple Rule : Flagged as Decaying = True
  - Ground Truth: Site Seasonality
  - ML Advantage: FALSE POSITIVE (Wasted refresh effort)

[Content ID]: page_2_real_decay
  - Metrics     : Click Change: -25%, Position Change: +4.2
  - Simple Rule : Flagged as Decaying = True
  - Ground Truth: Content Decay / Outdated
  - ML Advantage: Correctly identified multi-feature signal

[Content ID]: page_3_position_drop
  - Metrics     : Click Change: -10%, Position Change: +3.5
  - Simple Rule : Flagged as Decaying = False
  - Ground Truth: Algorithm / Ranking Shift
  - ML Advantage: Correctly identified multi-feature signal


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.